# DPM Engine · Parameterisation Load

**Purpose.** Makes a parameterisation file uploaded with the DataLoader the current parameterisation of its framework, only if it is valid. It is run by the batch `DPM_Param_Load`, which each of the four DataLoaders of the file (Datapoints, Filters, Catalog, Sources) triggers when its load ends.

- Looks for uploads that are complete (the four tabs loaded) and new (not processed before). If an upload is still arriving, it waits for its missing tabs (`upload_wait_seconds`), because the orchestrator skips the triggers that arrive while the batch is running. Without any new upload, it ends without doing anything.
- Validates each upload in the raw tables with `DE_ENR_NB_LGL_REG_DPM_METADATA_VALIDATION` (candidate mode).
- `VALID`: the upload replaces the parameterisation of its framework in the four SCD 2.2 tables. Rows are compared by content: unchanged rows keep their version, new rows start a version and rows that are no longer in the file are closed. No other framework is touched. The verdict is recorded with the snapshot the upload creates.
- `INVALID`: the parameterisation is not changed; the issues are in `dp_lgl_reg_dpm_iss_log` and the notebook ends with an error (`fail_on_invalid_upload`).
- Every upload processed is recorded in `dp_lgl_reg_dpm_param_load`.

**Sources:** `raw_dpm_param_datapoint`, `raw_dpm_param_fltr`, `raw_dpm_param_ctlg`, `raw_dpm_param_src_tbl` (DataLoader, append).

**Sink tables:** `dp_lgl_reg_dpm_param_datapoint`, `dp_lgl_reg_dpm_param_fltr`, `dp_lgl_reg_dpm_param_ctlg`, `dp_lgl_reg_dpm_param_src_tbl`, `dp_lgl_reg_dpm_param_vld`, `dp_lgl_reg_dpm_param_load`.

**Parameters:** table names, `validation_notebook`, `validation_timeout_seconds`, `upload_wait_seconds`, `upload_poll_seconds`, `fail_on_invalid_upload`. The batch passes none.

**Prerequisite:** the engine tables exist (`DE_DP_NB_LGL_REG_CREATE_TABLES`, run at deployment). The name of the file identifies the framework (`DPM_Param_<FRAMEWORK>.xlsx`): a file name is never reused for another framework.

**Output:** the notebook ends with the result of each upload as its exit value (JSON).

**Location:** workspace *DP FET Legal and Regulatory Reporting*, folder `01_Processes/DPM`, default Lakehouse `DP_FET_LH_LegalAndRegulatoryReporting`.

**Changelog:** 2026-11-11 | Notebook creation.


## 1. Parameters


In [ ]:
# Raw tables loaded by the DataLoader (one per tab of the parameterisation file).
raw_datapoints_table = "raw_dpm_param_datapoint"
raw_filters_table = "raw_dpm_param_fltr"
raw_catalog_table = "raw_dpm_param_ctlg"
raw_source_config_table = "raw_dpm_param_src_tbl"

# Parameterisation tables of the engine (SCD 2.2).
datapoints_table = "dp_lgl_reg_dpm_param_datapoint"
filters_table = "dp_lgl_reg_dpm_param_fltr"
catalog_table = "dp_lgl_reg_dpm_param_ctlg"
source_config_table = "dp_lgl_reg_dpm_param_src_tbl"

# Verdicts and log of the uploads processed.
validation_table = "dp_lgl_reg_dpm_param_vld"
load_log_table = "dp_lgl_reg_dpm_param_load"

# Notebook that validates each upload, and the maximum time it may take.
validation_notebook = "DE_ENR_NB_LGL_REG_DPM_METADATA_VALIDATION"
validation_timeout_seconds = 3600

# The orchestrator skips a trigger of the batch while the batch is running, so the run
# that finds an upload still arriving waits for its missing tabs: it looks again every
# upload_poll_seconds, for upload_wait_seconds at most.
upload_wait_seconds = 600
upload_poll_seconds = 30

# True = the notebook ends with an error when an upload is INVALID, after recording it,
# so that the batch shows it. False once the result is notified by e-mail.
fail_on_invalid_upload = True


StatementMeta(, f856dd06-5b4b-4704-98fd-ee7add886977, 20, Finished, Available, Finished, False)

## Common definitions

Helpers shared with the other notebooks of the engine (`DE_ENR_NB_LGL_REG_DPM_COMMON`).


In [ ]:
%run DE_ENR_NB_LGL_REG_DPM_COMMON

StatementMeta(, f856dd06-5b4b-4704-98fd-ee7add886977, 21, Finished, Available, Finished, True)

## 2. Uploads, promotion and log


In [ ]:
import json
import time

from delta.tables import DeltaTable
from notebookutils import mssparkutils
from pyspark.sql import Window

# Tab of the parameterisation file, raw table and parameterisation table, in load order.
TABS = (
    ("datapoints", "Datapoints", raw_datapoints_table, datapoints_table),
    ("filters", "Filters", raw_filters_table, filters_table),
    ("catalog", "Catalog", raw_catalog_table, catalog_table),
    ("sources", "Sources", raw_source_config_table, source_config_table),
)
# Columns of the load log with the dataload_id of each tab.
LOAD_ID_COLUMNS = {
    "datapoints": "datapoint_load_id",
    "filters": "fltr_load_id",
    "catalog": "ctlg_load_id",
    "sources": "src_load_id",
}
# Columns that the DataLoader adds to the raw tables and that are not parameterisation.
RAW_TECHNICAL_COLUMNS = {"dataload_id", "ingestion_timestamp", "usr_id"}
LOAD_LOG_SCHEMA = (
    "src_file_nm STRING, framework_cd STRING, datapoint_load_id STRING, fltr_load_id STRING, "
    "ctlg_load_id STRING, src_load_id STRING, load_usr_id STRING, load_sts_cd STRING, "
    "vld_id STRING, param_snpsh_dt TIMESTAMP, crtn_dt TIMESTAMP"
)
VALIDATION_SCHEMA = (
    "vld_id STRING, framework_cd STRING, param_snpsh_dt TIMESTAMP, "
    "vld_stts STRING, err_nr BIGINT, crtn_dt TIMESTAMP"
)
ROW_KEY = ["row_hsh_cd", "row_occ_nr"]


def latest_loads(table_name, tab):
    """Latest load of each file in a raw table: its dataload_id, moment and user."""
    loads = (
        spark.table(table_name)
        .groupBy(F.trim("src_file_nm").alias("src_file_nm"), "dataload_id")
        .agg(F.max("ingestion_timestamp").alias("load_dt"), F.max("usr_id").alias("usr_id"))
    )
    latest = Window.partitionBy("src_file_nm").orderBy(F.col("load_dt").desc(), F.col("dataload_id").desc())
    return (
        loads.withColumn("load_rank_nr", F.row_number().over(latest))
        .where(F.col("load_rank_nr") == 1)
        .select(
            "src_file_nm",
            F.col("dataload_id").alias(f"{tab}_load_id"),
            F.col("load_dt").alias(f"{tab}_load_dt"),
            F.col("usr_id").alias(f"{tab}_usr_id"),
        )
    )


def uploads_state():
    """Uploads to process, oldest first, and number of uploads that are still arriving.

    An upload is the latest load of the four tabs of one file. It is ready when the four
    raw tables have rows of the file and none of its four loads has been processed
    before. A file with only some new loads is still arriving if those loads are recent
    (within upload_wait_seconds of the latest load of the raw tables); an older one is a
    partial upload that was never completed, and it is not waited for.
    """
    latest = None
    for tab, _, raw_table, _ in TABS:
        loads = latest_loads(raw_table, tab)
        latest = loads if latest is None else latest.join(loads, "src_file_nm", "full")
    files = [row.asDict() for row in latest.toLocalIterator()]
    processed = {tab: set() for tab in LOAD_ID_COLUMNS}
    for row in spark.table(load_log_table).select(*LOAD_ID_COLUMNS.values()).toLocalIterator():
        for tab, column in LOAD_ID_COLUMNS.items():
            processed[tab].add(row[column])
    newest = max((row[f"{tab}_load_dt"] for row in files for tab in LOAD_ID_COLUMNS if row[f"{tab}_load_dt"]), default=None)
    ready, arriving = [], 0
    for row in files:
        loads = {tab: row[f"{tab}_load_id"] for tab in LOAD_ID_COLUMNS}
        new_moments = [
            row[f"{tab}_load_dt"] for tab in loads
            if loads[tab] is not None and loads[tab] not in processed[tab]
        ]
        if len(new_moments) == len(loads):
            ready.append({
                "src_file_nm": row["src_file_nm"],
                "loads": loads,
                "load_usr_id": clean(row["datapoints_usr_id"]),
                "load_dt": max(new_moments),
            })
        elif new_moments and (newest - max(new_moments)).total_seconds() <= int(upload_wait_seconds):
            arriving += 1
    return sorted(ready, key=lambda upload: upload["load_dt"]), arriving


def with_row_key(frame, columns):
    """Adds the key that identifies a parameterisation row: the hash of its content and,
    for rows with the same content, their occurrence number."""
    content = F.to_json(F.struct(*[F.coalesce(F.col(name), F.lit("")).alias(name) for name in columns]))
    occurrence = Window.partitionBy("row_hsh_cd").orderBy(F.lit(1))
    return (
        frame.withColumn("row_hsh_cd", F.sha2(content, 256))
        .withColumn("row_occ_nr", F.row_number().over(occurrence))
    )


def plan_promotion(target_table, incoming, scope_files, tab_label, dataload_id, version_ts):
    """Changes that leave the rows of a framework in a parameterisation table equal to an upload.

    The rows of the upload are compared by content with the current rows of the files of
    the framework. Returns the new versions to insert (rows of the upload that are not
    current, with the schema of the table) and the actions on the current rows, by
    surrogate key: CLOSE for the rows that are no longer in the upload and SEEN for the
    rows that stay. A row whose content has not changed keeps its version.
    """
    target = spark.table(target_table)
    key_column = next(name for name in target.columns if name.startswith("sk_"))
    business = [
        name for name in target.columns
        if name in incoming.columns and name not in RAW_TECHNICAL_COLUMNS
    ]
    uploaded = with_row_key(
        incoming.select(*[F.coalesce(F.col(name), F.lit("")).alias(name) for name in business]), business
    )
    current = with_row_key(
        target.where((F.col("ctrl_state") == 1) & (F.col("row_present_flg") == 1))
        .where(F.col("src_file_nm").isin(scope_files)),
        business,
    )
    version = F.lit(version_ts).cast("timestamp")
    technical = {
        "src_she_nm": F.lit(tab_label),
        "metadata_load_id": F.lit(dataload_id),
        "load_dt": version,
        "row_present_flg": F.lit(1),
        "dpm_vrs_dt": version,
        "dpm_ingestion_dt": version,
        "ref_start_dt": version,
        "ref_last_dt": version,
        "ctrl_start_dt": version,
        "ctrl_updt_dt": version,
        "ctrl_state": F.lit(1),
    }
    last_key = target.agg(F.max(key_column)).first()[0] or 0
    new_rows = uploaded.join(current.select(*ROW_KEY), ROW_KEY, "left_anti").withColumn(
        key_column, F.lit(last_key) + F.row_number().over(Window.orderBy(*ROW_KEY))
    )
    new_versions = new_rows.select(*[
        (
            F.col(field.name) if field.name in business + [key_column]
            else technical.get(field.name, F.lit(None))
        ).cast(field.dataType).alias(field.name)
        for field in target.schema.fields
    ])
    actions = (
        current.join(uploaded.select(*ROW_KEY, F.lit("SEEN").alias("act_cd")), ROW_KEY, "left")
        .select(F.col(key_column).alias("row_sk"), F.coalesce(F.col("act_cd"), F.lit("CLOSE")).alias("act_cd"))
    )
    # Fixed before the table is written: the changes are computed on its current content.
    return new_versions.localCheckpoint(eager=True), actions.localCheckpoint(eager=True)


def apply_promotion(target_table, new_versions, actions, version_ts):
    """Writes the changes of plan_promotion(): closes the versions that are no longer in the
    upload, marks the ones that stay as seen and inserts the new ones. Returns the number
    of versions closed or inserted."""
    key_column = next(name for name in new_versions.columns if name.startswith("sk_"))
    version = F.lit(version_ts).cast("timestamp")
    write_with_retry(
        lambda: (
            DeltaTable.forName(spark, target_table)
            .alias("t")
            .merge(actions.alias("s"), F.col(f"t.{key_column}") == F.col("s.row_sk"))
            .whenMatchedUpdate(
                condition=F.col("s.act_cd") == "CLOSE",
                set={"ref_end_dt": version, "ctrl_end_dt": version, "ctrl_updt_dt": version, "ctrl_state": F.lit(0)},
            )
            .whenMatchedUpdate(set={"ref_last_dt": version, "ctrl_updt_dt": version})
            .execute()
        )
    )
    new_versions.write.format("delta").mode("append").saveAsTable(target_table)
    return actions.where(F.col("act_cd") == "CLOSE").count() + new_versions.count()


def promote_upload(upload, framework, version_ts):
    """Makes a validated upload the current parameterisation of its framework.

    The four tables are compared with the rows of every file of the framework (the file of
    the upload and any file the framework was loaded from before), so the upload replaces
    the framework whole and no other framework is touched. Returns the number of versions
    closed or inserted; 0 when the upload is identical to the current parameterisation.
    """
    frames = dict(zip(("datapoints", "filters", "catalog", "sources"), candidate_frames(upload["loads"])))
    current_files = [
        row["src_file_nm"]
        for row in spark.table(datapoints_table)
        .where((F.col("ctrl_state") == 1) & (F.col("row_present_flg") == 1))
        .where(F.upper(F.trim("framework_cd")) == framework)
        .select("src_file_nm").distinct().toLocalIterator()
    ]
    scope_files = sorted(set(current_files) | {upload["src_file_nm"]})
    changes = 0
    for tab, tab_label, _, target_table in TABS:
        new_versions, actions = plan_promotion(
            target_table, frames[tab], scope_files, tab_label, upload["loads"][tab], version_ts
        )
        changes += apply_promotion(target_table, new_versions, actions, version_ts)
    return changes


def validate_upload(upload):
    """Runs the validation notebook on the upload (candidate mode); returns its summary."""
    summary = mssparkutils.notebook.run(
        validation_notebook,
        int(validation_timeout_seconds),
        {"candidate_json": json.dumps(upload["loads"]), "fail_notebook_on_invalid_metadata": False},
    )
    return json.loads(summary)


def record_verdict(validation, snapshot_ts):
    """Records the VALID verdict of a promoted upload, for the snapshot it is current in."""
    rows = [
        {
            "vld_id": validation["vld_id"], "framework_cd": framework, "param_snpsh_dt": snapshot_ts,
            "vld_stts": status, "err_nr": 0, "crtn_dt": utc_now(),
        }
        for framework, status in validation["verdicts"].items()
    ]
    spark.createDataFrame(rows, VALIDATION_SCHEMA).write.format("delta").mode("append").saveAsTable(validation_table)


def record_upload(upload, validation, status, snapshot_ts):
    """Adds the upload to the load log: it is not processed again."""
    row = {
        "src_file_nm": upload["src_file_nm"],
        "framework_cd": ",".join(validation["frameworks"]),
        **{column: upload["loads"][tab] for tab, column in LOAD_ID_COLUMNS.items()},
        "load_usr_id": upload["load_usr_id"],
        "load_sts_cd": status,
        "vld_id": validation["vld_id"],
        "param_snpsh_dt": snapshot_ts,
        "crtn_dt": utc_now(),
    }
    spark.createDataFrame([row], LOAD_LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(load_log_table)


def process_upload(upload):
    """Validates an upload and, if it is VALID, promotes it; returns its result.

    An INVALID upload is recorded and leaves the parameterisation as it was. The verdict of
    a promoted upload is recorded with the snapshot it creates, or with the current one if
    the upload is identical to the current parameterisation.
    """
    validation = validate_upload(upload)
    valid = bool(validation["verdicts"]) and not validation["invalid_frameworks"]
    snapshot_ts = None
    if valid:
        version_ts = utc_now()
        changes = promote_upload(upload, validation["frameworks"][0], version_ts)
        snapshot_ts = version_ts if changes else resolve_metadata_as_of("latest")
        record_verdict(validation, snapshot_ts)
    status = "VALID" if valid else "INVALID"
    record_upload(upload, validation, status, snapshot_ts)
    return {
        "src_file_nm": upload["src_file_nm"],
        "load_usr_id": upload["load_usr_id"],
        "frameworks": validation["frameworks"],
        "load_sts_cd": status,
        "vld_id": validation["vld_id"],
        "issue_count": validation["issue_count"],
        "param_snpsh_dt": str(snapshot_ts or ""),
    }


def run_param_load():
    """Processes the uploads that are ready; returns the result of each one.

    The DataLoader triggers the batch once per tab, and the orchestrator skips the triggers
    that arrive while the batch is running. So this run does not end while an upload is
    still arriving: it waits for the missing tabs (upload_wait_seconds at most) and
    processes the upload itself. Without any new upload it ends without doing anything.
    """
    results = []
    deadline = time.monotonic() + int(upload_wait_seconds)
    while True:
        ready, arriving = uploads_state()
        results += [process_upload(upload) for upload in ready]
        if not arriving or time.monotonic() >= deadline:
            break
        time.sleep(int(upload_poll_seconds))
    print(json.dumps(results, indent=2))
    invalid = [result["src_file_nm"] for result in results if result["load_sts_cd"] == "INVALID"]
    if invalid and bool(fail_on_invalid_upload):
        raise RuntimeError(
            f"Upload(s) not valid, parameterisation not changed: {invalid}. "
            "See the issues of each validation in the issue log."
        )
    return results


StatementMeta(, f856dd06-5b4b-4704-98fd-ee7add886977, 22, Finished, Available, Finished, False)

## 3. Execute load


In [ ]:
param_load_result = run_param_load()
# Exit value for the batch that launches this notebook (result of each upload, as JSON).
mssparkutils.notebook.exit(json.dumps(param_load_result))


StatementMeta(, f856dd06-5b4b-4704-98fd-ee7add886977, 23, Finished, Available, Finished, False)

[
  {
    "src_file_nm": "DPM_Param_FINREP.xlsx",
    "load_usr_id": "laura.martinez@novobanco.pt",
    "frameworks": [
      "FINREP"
    ],
    "load_sts_cd": "VALID",
    "vld_id": "210fa130-b507-4b3e-9e43-915e1e9fcd67",
    "issue_count": 0,
    "param_snpsh_dt": "2026-11-17 11:15:53.690788"
  }
]
ExitValue: [{"src_file_nm": "DPM_Param_FINREP.xlsx", "load_usr_id": "laura.martinez@novobanco.pt", "frameworks": ["FINREP"], "load_sts_cd": "VALID", "vld_id": "210fa130-b507-4b3e-9e43-915e1e9fcd67", "issue_count": 0, "param_snpsh_dt": "2026-11-17 11:15:53.690788"}]